In [1]:
from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from plotstyle import apply_paper_style
apply_paper_style()

In [1]:
from plot_spearman import plot_spearman

In [3]:
pwd

'/Users/sasthana/curnagl_shivanshi/Downscaling/Processing_and_Analysis_Scripts/Analysis/BCSR_Stats'

In [4]:
plot_spearman(
in_csv= "Tables/intervariable_spearman_ensmeans_2015_2023.csv",
out_pdf= "Figures/Spearman.pdf",
dpi=400,
)
plt.show()

[ok] wrote Figures/Spearman.pdf


In [22]:
pwd

'/Users/sasthana/curnagl_shivanshi/Downscaling/Processing_and_Analysis_Scripts/Analysis/BCSR_Stats'

In [ ]:
import seaborn as sns

In [27]:
from seaborn import heatmap
%matplotlib inline

In [28]:
autocorr_table = pd.read_csv("Tables/autocorrelation_monthly_mean_ensmeans_tas_2015_2023.csv", index_col=0).astype(float)

Trends 

In [30]:
pr_trend = "Tables/trend_pr_1981_2010_to_2070_2099.csv"
tas_trend = "Tables/trend_tas_1981_2010_to_2070_2099.csv"

In [31]:
baseline_label = "Coarse"

MODEL_ORDER = [
    "EQM + Bilinear",
    "CDF-t + Bilinear",
    "dOTC + Bilinear",
    "EQM + Bilinear + U-Net",
    "CDF-t + Bilinear + U-Net",
    "dOTC + Bilinear + U-Net",
    "EQM + Bilinear + U-Net + DDIM",
    "CDF-t + Bilinear + U-Net + DDIM",
    "dOTC + Bilinear + U-Net + DDIM",
    "CH2025 methodological baseline",
]

In [35]:
def _read_trend_table(path):
    df = pd.read_csv(path)
    if "Unnamed: 0" in df.columns:
        df = df.rename(columns={"Unnamed: 0": "method"})
    elif "method" not in df.columns:
        df = df.rename(columns={df.columns[0]: "method"})
    return df

def _pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"None of {candidates} found in {list(df.columns)}")

def _prepare_series(df, trend_candidates):
    c = _pick_col(df, trend_candidates)
    s = df.set_index("method")[c].astype(float)

    if baseline_label in s.index:
        bname = baseline_label
    else:
        matches = [i for i in s.index if ("baseline" in i.lower()) or ("coarse" in i.lower())]
        bname = matches[0]

    b = float(s.loc[bname])
    d = s - b
    return s, d, bname, b, c

pr_df = _read_trend_table(pr_trend)
tas_df = _read_trend_table(tas_trend)



In [22]:
pr_raw, pr_delta, pr_bname, pr_b, pr_col = _prepare_series(pr_df, ["trend_change_pct", "trend_change"])
tas_raw, tas_delta, tas_bname, tas_b, tas_col = _prepare_series(tas_df, ["trend_change_abs", "trend_change"])

models = [m for m in MODEL_ORDER if (m in tas_delta.index) or (m in pr_delta.index)]
tas_delta = tas_delta.reindex(models)
pr_delta = pr_delta.reindex(models)

y = np.arange(len(models))

In [ ]:
from plotstyle import apply_paper_style, style_axis, save_figure, get_model_color

def _model_family(name: str) -> str:
    n = name.lower()
    if "ddim" in n:
        return "DDIM"
    if "u-net" in n or "unet" in n:
        return "UNet"
    if "baseline" in n or "coarse" in n or "ch2025" in n:
        return "Coarse"
    return "Bilinear"

apply_paper_style()

pr_raw, pr_delta, pr_bname, pr_b, pr_col = _prepare_series(pr_df, ["trend_change_pct", "trend_change"])
tas_raw, tas_delta, tas_bname, tas_b, tas_col = _prepare_series(tas_df, ["trend_change_abs", "trend_change"])

models = [m for m in MODEL_ORDER if (m in tas_delta.index) or (m in pr_delta.index)]
tas_delta = tas_delta.reindex(models)
pr_delta = pr_delta.reindex(models)

y = np.arange(len(models))
fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)

bar_colors = [get_model_color(_model_family(m)) for m in models]

axes[0].barh(y, tas_delta.values, color=bar_colors, alpha=0.9)
axes[0].axvline(0, color="yellow", lw=4)
axes[0].set_yticks(y)
axes[0].set_yticklabels(models)
style_axis(axes[0], xlabel="Δ (°C) relative to coarse RCM", grid=False, title="(a) Temperature")
axes[0].grid(False) 

axes[1].barh(y, pr_delta.values, color=bar_colors, alpha=0.9)
axes[1].axvline(0, color="yellow", lw=4)
axes[1].set_yticks(y)
axes[1].set_yticklabels(models)
style_axis(axes[1], xlabel="Δ (%) relative to coarse RCM", grid=False, title="(b) Precipitation")
axes[1].grid(False)

axes[0].invert_yaxis()
plt.tight_layout()
save_figure(fig, "Figures/Trend.pdf")
plt.show()